## Why Semantic Caching?

Every call to `handle_query()` costs at least one LLM call for routing (`route_query`), and on top of that, retrieval plus another LLM call for generation. In production, a large share of incoming queries are near-duplicates of ones already answered — e.g. "What was Uber's 2023 revenue?" vs. "What did Uber make in 2023?" — so re-running the full pipeline for each rephrasing wastes both latency and cost.

A semantic cache sits in front of the pipeline: it embeds the incoming query, checks a vector store of past query/answer pairs for a near-neighbor above a similarity threshold, and returns the cached answer on a hit instead of calling the router, retriever, and generator again. It only helps for repeated-in-spirit questions — a genuinely novel query still falls through to the full pipeline.

## Architecture

![Semantic cache and routing architecture](../assets/ch07__image005.png)

The diagram splits the system into two foundational pillars:

**Semantic Cache (left)**
- The incoming query is checked by a fast **keyword-based filter** for time-sensitivity (words like "today," "latest," "current") and, optionally, a slower **LLM-based classifier** for harder cases.
- **Cache Lookup** only returns a hit if similarity is high *and* the query isn't time-sensitive — a textually similar query is still rejected if it's asking about something that could have changed (e.g. "Uber's latest revenue" shouldn't reuse a stale cached answer).
- A match pulls from **Stored Results** (vector + text) and returns a **Cached Response**.

**Routing (right)**
- No match, or a time-sensitive query, falls through to the same **Retriever → Generator** pipeline used by `handle_query()`, producing a **Fresh Response**.

Both paths converge into the same **System Output** — callers don't need to know whether an answer came from cache or from a fresh pipeline run. (Not shown: after a fresh response is generated, it should be written back into Stored Results so the next similar query becomes a cache hit.)

In [ ]:
import os

from dotenv import load_dotenv
from qdrant_client import AsyncQdrantClient

from enterprise_rag.semantic_cache import SemanticCaching, ask

load_dotenv()

qdrant_url = os.getenv("QDRANT_URL")
if qdrant_url:
    qdrant = AsyncQdrantClient(url=qdrant_url, api_key=os.getenv("QDRANT_API_KEY"))
else:
    qdrant = AsyncQdrantClient(location=":memory:")

In [ ]:
import time

cache = SemanticCaching(clear_on_init=True)

# First call: cache miss, full RAG pipeline
start = time.time()
answer_1 = await ask(cache, qdrant, "What was Uber's revenue in 2021?")
miss_time = time.time() - start

# Second call: semantically similar query, cache hit
start = time.time()
answer_2 = await ask(cache, qdrant, "How much did Uber earn in fiscal year 2021?")
hit_time = time.time() - start

print(f"Cache MISS: {miss_time:.2f}s")
print(f"Cache HIT:  {hit_time:.3f}s")
print(f"Speedup:    {miss_time / hit_time:.0f}x")